# How to use vertical interpolation for fieldlists  with multiple dates?

The vertical interpolation for fieldlists only works for one set of unique levels. If your data contains the same set of levels for multiple dates/ensemble members etc. you need to process it in a loop.

We use GRIB data containing the same pressure levels for 4 different base datetimes.

In [1]:
import earthkit.data as ekd

from earthkit.meteo.vertical import fieldlist as vertical

# get GRIB fieldlist
fl = ekd.from_source("sample", "era5_tquv_pl_subarea.grib").to_fieldlist()

t = fl.sel({"parameter.variable": "t"}).order_by("time.base_datetime")
t.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,700,pressure,0,regular_ll
1,t,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,850,pressure,0,regular_ll
2,t,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,925,pressure,0,regular_ll
3,t,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,700,pressure,0,regular_ll
4,t,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,850,pressure,0,regular_ll
5,t,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,925,pressure,0,regular_ll
6,t,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,700,pressure,0,regular_ll
7,t,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,850,pressure,0,regular_ll
8,t,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,925,pressure,0,regular_ll
9,t,2016-09-26 18:00:00,2016-09-26 18:00:00,0 days,700,pressure,0,regular_ll


In [2]:
# extract the unique dates
dates = t.unique("time.base_datetime")
dates

{'time.base_datetime': (datetime.datetime(2016, 9, 26, 0, 0),
  datetime.datetime(2016, 9, 26, 6, 0),
  datetime.datetime(2016, 9, 26, 12, 0),
  datetime.datetime(2016, 9, 26, 18, 0))}

In [3]:
# create empty fieldlist
t_res = ekd.create_fieldlist()

# interpolate in a loop
for d in dates["time.base_datetime"]:
    t_d = t.sel({"time.base_datetime": d})
    coords = [f.vertical.level(units="Pa") for f in t_d]  # Pa coord
    target_coords = [90000, 80000]  # Pa

    # we can only pass one set of unique levels for this method
    r = vertical.interpolate_monotonic(t_d, coords=coords, target_coords=target_coords, coord_type="pressure")

    t_res = ekd.concat(t_res, r)

t_res.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,900.0,pressure,0,regular_ll
1,t,2016-09-26 00:00:00,2016-09-26 00:00:00,0 days,800.0,pressure,0,regular_ll
2,t,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,900.0,pressure,0,regular_ll
3,t,2016-09-26 06:00:00,2016-09-26 06:00:00,0 days,800.0,pressure,0,regular_ll
4,t,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,900.0,pressure,0,regular_ll
5,t,2016-09-26 12:00:00,2016-09-26 12:00:00,0 days,800.0,pressure,0,regular_ll
6,t,2016-09-26 18:00:00,2016-09-26 18:00:00,0 days,900.0,pressure,0,regular_ll
7,t,2016-09-26 18:00:00,2016-09-26 18:00:00,0 days,800.0,pressure,0,regular_ll
